# 04. Evaluation, uncertainty and report evidence

UTS 43008 · Assignment 3 · Project 12 · Dihedrals

The question is whether PA improves over P, not whether one replay looks convincing.
Evaluate matched learners on identical layouts and views. Keep training, development and final
evidence separate.

This notebook neither trains nor runs final evaluation. It inspects an explicitly selected CSV.
Its small numerical example tests the statistics only; it is not a project result.

In [ ]:
from pathlib import Path
import os
import sys
import json
import uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidate = Path(os.environ.get("BASE_SIEGE_ROOT", Path.cwd())).expanduser()
ROOT = next((p for p in [candidate, *candidate.parents] if (p / "env/siege_env.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the complete project, or set BASE_SIEGE_ROOT to its folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from env import config as C
from env.siege_env import SiegeEnv, describe_action, deployment_action
from env.render import draw_snapshot
from env.generator import generate_base
from experiments.provenance import runtime_manifest, sha256
RUNS = Path(os.environ.get("BASE_SIEGE_RUNS_DIR", ROOT / "runs")).expanduser().resolve()
OUTPUT = ROOT / "artifacts" / "notebook-sessions" / uuid.uuid4().hex[:12]
OUTPUT.mkdir(parents=True, exist_ok=False)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 12, "axes.labelsize": 10, "font.size": 10})
def save_figure(name):
    plt.savefig(OUTPUT / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.show()
print(f"Project: {ROOT.name} | Python: {sys.version.split()[0]}")
print(f"This notebook's outputs: {OUTPUT}")

## 1. The evaluation unit

One row is one policy on one layout and view. A layout is identified by seed and structure.
Each independent learner seed needs both P and PA, with matched settings, source, pools and
actual phase budgets.

Games from the same learner are correlated. Thousands of games are not thousands of independent
learners. Average matched differences within each learner, then estimate uncertainty across
independent learner seeds.

In [ ]:
from train.evaluate import paired_summary, study_summary, validate_matched_runs, load_run

display(pd.DataFrame([
    ["condition", "P, PA, heuristic or random"], ["learner_seed", "Independent training run"],
    ["base_seed + motif + view", "Matched evaluation situation"],
    ["destruction", "Fraction of non-wall value destroyed"],
    ["town_hall / full_clear", "Binary outcomes"], ["return", "Reward including milestones and cost"],
    ["evidence_scope", "Development, software check or authorised final"]
], columns=["field", "meaning"]))

## 2. Select actual evidence

Set the CSV path after development evaluation. Leave it unset until results exist.
Do not automatically pick the newest file; that can mix a smoke check with study evidence.

In [ ]:
EVALUATION_CSV = None
frame = None
if EVALUATION_CSV is not None:
    frame = pd.read_csv(Path(EVALUATION_CSV))
    required = {"condition", "learner_seed", "base_seed", "motif", "view",
                "destruction", "town_hall", "full_clear", "return", "evidence_scope"}
    assert required.issubset(frame.columns), f"Missing: {required - set(frame.columns)}"
    display(frame.groupby(["evidence_scope", "condition"]).size().rename("episodes"))
    display(frame.head())
else:
    print("No results selected. No policy-performance claim is available.")

## 3. Check provenance before a headline

Validate the corresponding run folders, not just the CSV. A CSV alone cannot prove equal training
budgets, source or checkpoint provenance. Retain every raw episode row and report accepted versus
candidate counts and coverage by structure.

In [ ]:
RUN_FOLDERS = []
if RUN_FOLDERS:
    manifests = [load_run(Path(path))[1] for path in RUN_FOLDERS]
    print("Matched independent learner seeds:", validate_matched_runs(manifests))
else:
    print("No run folders selected for provenance validation.")

## 4. Understand the paired interval

For each learner, average PA destruction minus P destruction over matched evaluation situations.
The headline is the mean of those learner differences. The Student-t interval uses their sample
standard deviation and the number of independent learners.

The five differences below are invented values for testing the estimator. They are not saved
as project results. With only one learner pair, the implementation deliberately returns no interval.

In [ ]:
toy_differences = [.01, .04, -.02, .03, .02]
toy_rows = []
for learner, difference in enumerate(toy_differences):
    for base in range(3):
        for condition in ["P", "PA"]:
            toy_rows.append({"condition": condition, "learner_seed": learner, "base_seed": base,
                "motif": "numerical_example", "view": 0,
                "destruction": .4 + (difference if condition == "PA" else 0)})
toy = pd.DataFrame(toy_rows)
toy_summary = paired_summary(toy)
display(pd.Series(toy_summary, name="Numerical example only"))
assert toy_summary["n_independent_learners"] == 5
assert np.isclose(toy_summary["mean_paired_difference"], np.mean(toy_differences))
assert paired_summary(toy.query("learner_seed == 0"))["ci95_low"] is None

## 5. Analyse the real paired results

A positive difference favours PA. If the interval includes zero, the evidence does not establish
a consistent direction at that confidence level. Even an interval above zero does not prove the
method works in all games or beyond the filtered study population.

In [ ]:
if frame is not None and {"P", "PA"}.issubset(set(frame.condition)):
    summary = paired_summary(frame)
    display(pd.Series(summary))
    differences = pd.Series(summary["learner_differences"], dtype=float)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.axhline(0, color="grey", linewidth=1)
    ax.scatter(differences.index, differences.values)
    ax.set(xlabel="Independent learner seed", ylabel="Mean destruction difference: PA − P",
           title=", ".join(sorted(frame.evidence_scope.unique())))
    save_figure("paired-learner-differences")
else:
    print("Real paired analysis awaits a P/PA evaluation CSV.")

## 6. Transfer and orientation

Report training structures and layered-core transfer separately. Transfer is absent from training
and current development evaluation. Earlier software checks included this structure, so a claim
that it was never inspected during development would be inaccurate.

For each learner and structure, orientation gap is maximum minus minimum mean destruction across
eight views. A smaller gap suggests less sensitivity to presentation. It is meaningful only
when all eight views were evaluated.

In [ ]:
if frame is not None:
    means = frame.groupby(["condition", "motif", "view"]).destruction.mean().reset_index()
    display(means)
    view_counts = frame.groupby(["condition", "learner_seed", "base_seed", "motif"]).view.nunique()
    print("Every evaluation unit has eight views:", bool((view_counts == 8).all()))
    if {"P", "PA"}.issubset(set(frame.condition)):
        report = study_summary(frame)
        display(pd.DataFrame(report["orientation_gap"]))
else:
    print("Structure and orientation tables will use your selected CSV.")

## 7. Report-ready evidence

The experiment tools save raw episodes, paired summaries, structure summaries and provenance.
Notebook plots go into the printed output folder. Record that path when moving figures into
the report. Do not overwrite inconvenient results.

Include:

- Source/configuration and actual training budgets.
- Independent learner count, accepted layouts, views and random baseline action seeds.
- Main metric with learner-level uncertainty, plus Town Hall and full-clear outcomes.
- Structure results, orientation sensitivity and representative failures.
- Runtime details, interruptions and the filtered-population limitation.

## 8. Final evaluation is a separate gate

Finish development and every registered learner pair, then freeze verified artifacts.
The final command requires confirmation and binds one freeze to one output. This notebook never
invokes it. An interrupted final job can resume the same committed pool and episode journal;
different code, weights or output paths are rejected. Do not tune from final results, delete
the binding to obtain another attempt, or consume reserve seeds.

See [Runbook](../docs/RUNBOOK.md) for the registration, freeze and final workflow.

### Resuming development evaluation

The evaluator commits one complete episode at a time. If the process stops, add --resume to
the same command. Source, package versions, pool, model identities and views must still match.
A changed experiment needs a new output and must not reuse an old journal.

The following cell prints commands only. It does not load models, evaluate games or access
the final partition.

In [ ]:
import shlex
run_paths = [str(RUNS / "study-v2" / f"{condition}-{seed}")
             for seed in C.STUDY_LEARNER_SEEDS for condition in ("P", "PA")]
command = [sys.executable, "-m", "train.evaluate", "--pool", str(ROOT / "bases/dev-v2.json"),
           "--runs", *run_paths, "--all-views", "--out", str(ROOT / "artifacts/study-dev-v2")]
print("New evaluation:", shlex.join(command))
print("Compatible resume:", shlex.join(command + ["--resume"]))